# 03 — Baseline model

**What this baseline is:** a simple feed-forward network predicting temperature from
`latitude, longitude, depth` and calendar/seasonal features only. It does **not** see any
surface satellite inputs (SST, SSS, SLA, currents, winds) - it is essentially a learned
**climatology**: "what temperature does this place/depth/time of year usually have?"

**Why that's the right first baseline:** it's cheap to train and gives you a floor number.
If the real CNN+MLP model can't beat this, something is wrong with the surface-input pipeline,
not just the model architecture.

**Why it's not the final baseline:** the residual plot at the end of this notebook shows the
model over/under-predicting at temperature extremes - expected, since it has no way to see
day-to-day surface conditions, only average seasonal position. Before comparing against the
CNN, add a second baseline (e.g. Random Forest) that also takes SST/SSS/SLA/currents/winds as
input, so the comparison actually tests whether the CNN's use of surface patches adds value.

Fixes applied in this version vs. the original:
- Global random seeds, so results are reproducible.
- Train/val/test split is done **by date first, then sampled** - the original sampled 2M
  random rows first and sorted by time after, which doesn't guarantee clean date separation
  between splits. Splitting by date first guarantees no date appears in more than one split.
- Removed dead/duplicate code (features computed twice; an unused `X, y` from the full
  dataframe).
- Added float32 casting before `to_dataframe()` to reduce peak memory on the same machine
  that hit a `MemoryError` in preprocessing.
- Added **depth-resolved RMSE/MAE** (shallow / thermocline / deep), not just one overall
  number - this is what actually shows where the model is and isn't reliable.
- Saves the trained model, not just the scaler.

In [5]:
import os
import glob
import random
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

import joblib

# Reproducibility - the original notebook only seeded the sampling step,
# so the model itself trained differently on every run.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [6]:
DATA_DIR = "../data/processed"

nc_files = sorted(
    glob.glob(os.path.join(DATA_DIR, "*.nc"))
)

print("Files found:")
for file in nc_files:
    print(file)
print("\nNumber of files:", len(nc_files))

if len(nc_files) != 6:
    print("WARNING: Expected 6 NetCDF files.")

datasets = [xr.open_dataset(file) for file in nc_files]
print("All files loaded successfully.")

Files found:
../data/processed\glorys_2021_01_025deg.nc
../data/processed\glorys_2021_01_06_025deg_15depth_clean.nc
../data/processed\glorys_2021_01_06_025deg_clean.nc
../data/processed\glorys_2021_01_06_native_36depth_025deg.nc
../data/processed\glorys_2021_01_06_regridded_15depth.nc
../data/processed\glorys_2021_02_025deg.nc
../data/processed\glorys_2021_03_025deg.nc
../data/processed\glorys_2021_04_025deg.nc
../data/processed\glorys_2021_05_025deg.nc
../data/processed\glorys_2021_06_025deg.nc
../data/processed\ssh_currents_2021_01_06_regridded.nc
../data/processed\sss_2021_01_06_regridded.nc
../data/processed\sst_2021_01_06_regridded.nc
../data/processed\wind_metopb_2021_01_06_regridded.nc

Number of files: 14
All files loaded successfully.


In [7]:
ds = xr.concat(datasets, dim="time")

# Free the individual file handles now that everything is concatenated.
for d in datasets:
    d.close()

print("Combined dataset:")
print(ds)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13680\3407120480.py:1: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'depth' ('depth',) The recommendation is to set join explicitly for this case.
  ds = xr.concat(datasets, dim="time")


ValueError: 'depth' not present in all datasets and coords='different'. Either add 'depth' to datasets where it is missing or specify coords='minimal'.

In [ ]:
print("Dimensions:")
print(ds.dims)

print("\nCoordinates:")
print(ds.coords)

print("\nVariables:")
print(ds.data_vars)

In [ ]:
# Cast to float32 before converting to a DataFrame - to_dataframe() materializes
# every (time, depth, lat, lon) combination at once, so halving the per-value
# size here meaningfully lowers peak memory for this step.
thetao = ds["thetao"].astype("float32")

missing_count = thetao.isnull().sum().item()
print("Missing thetao values:", missing_count)
print("Mean:", thetao.mean().item())
print("Minimum:", thetao.min().item())
print("Maximum:", thetao.max().item())

In [ ]:
# Convert thetao to DataFrame
df = thetao.to_dataframe(name="thetao").reset_index()
print("Original shape:", df.shape)

# Keep only observations where thetao is available
df = df[df["thetao"].notna()].copy()
print("Shape after selecting valid thetao:", df.shape)
print("\nRemaining missing thetao:", df["thetao"].isna().sum())

# The full dataset is no longer needed once it's in df - free the memory
# before the feature engineering / sampling steps below.
del ds, thetao

In [ ]:
# Seasonal / calendar features. Computed once, here, on the full dataframe -
# the original notebook recomputed these a second time after sampling, which
# was redundant since sampling happens after this cell and preserves columns.
df["time"] = pd.to_datetime(df["time"])
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day_of_year"] = df["time"].dt.dayofyear
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

feature_columns = [
    "latitude",
    "longitude",
    "depth",
    "year",
    "day_of_year",
    "month_sin",
    "month_cos"
]
target_column = "thetao"

print("Features:", feature_columns)
print("\nMissing values in features:")
print(df[feature_columns].isna().sum())
print("\nMissing values in target:", df[target_column].isna().sum())

## Chronological split, done correctly

The original notebook randomly sampled 2M rows *first*, then sorted by time and split by row
position. With uniform sampling that often still lands close to a date-based split, but it
isn't guaranteed - and "close" isn't good enough when the whole point of the split is to
prevent the model from ever training on a date it's tested on.

Here the date cutoffs are computed **first**, from the full (unsampled) date range, and rows
are assigned to train/val/test by which side of those cutoffs their date falls on. Sampling
down to a manageable size happens *within* each split afterwards, so no date can ever appear
on both sides of a split.

In [ ]:
SAMPLE_SIZE = 2_000_000
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15  # remaining 0.15 is test

unique_dates = np.sort(df["time"].unique())
n_dates = len(unique_dates)

train_cutoff = unique_dates[int(n_dates * TRAIN_FRAC)]
val_cutoff = unique_dates[int(n_dates * (TRAIN_FRAC + VAL_FRAC))]

train_df = df[df["time"] < train_cutoff]
val_df = df[(df["time"] >= train_cutoff) & (df["time"] < val_cutoff)]
test_df = df[df["time"] >= val_cutoff]

print("Date ranges (no overlap by construction):")
print("Train:", train_df["time"].min(), "to", train_df["time"].max())
print("Val:  ", val_df["time"].min(), "to", val_df["time"].max())
print("Test: ", test_df["time"].min(), "to", test_df["time"].max())

# Sample down to a manageable training size, proportionally per split,
# now that each split is already confined to its own date range.
def sample_df(data, frac_of_total):
    n = min(len(data), int(SAMPLE_SIZE * frac_of_total))
    return data.sample(n=n, random_state=SEED)

train_df = sample_df(train_df, TRAIN_FRAC)
val_df = sample_df(val_df, VAL_FRAC)
test_df = sample_df(test_df, 1 - TRAIN_FRAC - VAL_FRAC)

print("\nTrain:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
X_train = train_df[feature_columns].values
y_train = train_df[target_column].values

X_val = val_df[feature_columns].values
y_val = val_df[target_column].values

X_test = test_df[feature_columns].values
y_test = test_df[target_column].values

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_val:  ", X_val.shape, " y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, " y_test: ", y_test.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")

In [ ]:
os.makedirs("../models", exist_ok=True)

joblib.dump(scaler, "../models/baseline_scaler.pkl")
print("Scaler saved.")

In [ ]:
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation="relu"),
    Dense(1, activation="linear")
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=1024,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
model.save("../models/baseline_ann.keras")
print("Model saved.")

In [ ]:
y_pred = model.predict(X_test_scaled, batch_size=1024).flatten()

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Baseline ANN Results (all depths combined)")
print("-------------------------------------------")
print("MSE :", mse)
print("RMSE:", rmse)
print("MAE :", mae)
print("R²  :", r2)

## Depth-resolved evaluation

A single overall RMSE hides whether the model is actually useful at depth, where the real
problem (subsurface reconstruction) matters most. Splitting into shallow / thermocline / deep
bands shows this directly, and is the same breakdown to use later for the CNN so the two are
comparable.

In [ ]:
depth_bins = [
    ("Shallow (0-100 m)", 0, 100),
    ("Thermocline (100-300 m)", 100, 300),
    ("Deep (300-1000 m)", 300, 1000),
]

test_depths = test_df["depth"].values

print(f"{'Band':<25}{'N':>10}{'RMSE':>10}{'MAE':>10}{'R²':>10}")
print("-" * 65)
band_results = []
for label, lo, hi in depth_bins:
    mask = (test_depths >= lo) & (test_depths <= hi)
    n = int(mask.sum())
    if n == 0:
        print(f"{label:<25}{n:>10}   (no test points in this band)")
        continue
    band_rmse = np.sqrt(mean_squared_error(y_test[mask], y_pred[mask]))
    band_mae = mean_absolute_error(y_test[mask], y_pred[mask])
    band_r2 = r2_score(y_test[mask], y_pred[mask])
    band_results.append((label, band_rmse))
    print(f"{label:<25}{n:>10}{band_rmse:>10.3f}{band_mae:>10.3f}{band_r2:>10.3f}")

plt.figure(figsize=(7, 4))
plt.bar([b[0] for b in band_results], [b[1] for b in band_results], color=["#065A82", "#E8734A", "#21295C"])
plt.ylabel("RMSE (°C)")
plt.title("Baseline ANN - RMSE by depth band")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(y_test, y_pred, s=5, alpha=0.3)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.xlabel("Actual Temperature")
plt.ylabel("Predicted Temperature")
plt.title("Actual vs Predicted Temperature")
plt.show()

In [ ]:
errors = y_test - y_pred
plt.figure(figsize=(8, 6))

plt.scatter(y_pred, errors, alpha=0.1)
plt.axhline(0, linestyle="--")

plt.xlabel("Predicted Temperature")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.show()

# This pattern - overpredicting at high temperatures, underpredicting at low
# ones - is expected for this baseline: it only sees latitude/longitude/depth/
# time, so it can only learn the *average* seasonal temperature at a location,
# not day-to-day surface conditions. A version of this baseline that also
# takes SST/SSS/SLA/currents/winds as input would be a fairer comparison
# point for the CNN, since it would have access to the same information.